In [1]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from xgboost.callback import EarlyStopping


df = pd.read_csv("investment_recommendations_10000.csv")

In [2]:
data_shape = df.shape
data_shape

(10000, 6)

In [3]:
data_info = df.info()
data_info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Individual Goals                 10000 non-null  object
 1   Age                              10000 non-null  int64 
 2   Gender                           10000 non-null  object
 3   Risk Tolerance                   10000 non-null  object
 4   Financial Literacy               10000 non-null  int64 
 5   Recommended Investment Products  10000 non-null  object
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


In [4]:
data_description= df.describe()
data_description

,Age,Financial Literacy
count,10000.00000,10000.000000
mean,39.84960,3.025500
std,23.27352,1.413807
min,0.00000,1.000000
25%,20.00000,2.000000
50%,40.00000,3.000000
75%,60.00000,4.000000
max,80.00000,5.000000


In [5]:
has_nulls = df.isnull().values.any()
has_nulls

False

In [6]:
column_names = df.columns
column_names

Index(['Individual Goals', 'Age', 'Gender', 'Risk Tolerance',
       'Financial Literacy', 'Recommended Investment Products'],
      dtype='object')

In [7]:
head = df.head()
head

,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy,Recommended Investment Products
0,saving for retirement,44,Male,Medium,2,"Mutual Funds, PPF, NPS, Stocks, Real Estate"
1,buying a house,47,Male,Medium,2,"Real Estate Investment Trusts (REITs), Home Lo..."
2,saving for education,64,Male,Low,4,"National Scholarship Scheme, Education Bonds, ..."
3,saving for retirement,67,Female,High,2,"Mutual Funds, PPF, NPS, Stocks, Real Estate"
4,saving for education,67,Male,High,3,"National Scholarship Scheme, Education Bonds, ..."


In [8]:
unique_element_count={}
unique_element={}
for column in column_names:
    count = df[column].nunique()
    unique_element_count[column] = (count)
    unique_elements = (count, df[column].unique())
unique_element_count, unique_elements

({'Individual Goals': 6,
  'Age': 81,
  'Gender': 2,
  'Risk Tolerance': 3,
  'Financial Literacy': 5,
  'Recommended Investment Products': 18},
 (18,
  array(['Mutual Funds, PPF, NPS, Stocks, Real Estate',
         'Real Estate Investment Trusts (REITs), Home Loans, Stocks, Real Estate',
         'National Scholarship Scheme, Education Bonds, Education Loans, Stocks, Mutual Funds',
         "Sukanya Samriddhi Yojana, Children's Mutual Funds, Real Estate Investment Trusts (REITs), Home Loans, Stocks",
         "Children's Mutual Funds, Personal Loans, Consumer Stocks, Gold, Mutual Funds",
         'Personal Loans, Consumer Stocks, Gold, Mutual Funds, Cryptocurrency',
         'Stand-Up India Scheme, Equity Shares, Business Loans, Stocks, Real Estate',
         "Children's Mutual Funds, National Scholarship Scheme, Education Bonds, Education Loans, Stocks",
         'Travel Insurance, Short-Term Mutual Funds, Savings Account, Stocks, Cryptocurrency',
         "Sukanya Samriddhi Yojana, 

In [9]:
investment_options=set()
for row in df['Recommended Investment Products']:
    investment=row.split(",")
    for i in range(len(investment)):
        investment_options.add(investment[i])
    
investment_options

{' Business Loans',
 " Children's Mutual Funds",
 ' Consumer Stocks',
 ' Cryptocurrency',
 ' Education Bonds',
 ' Education Loans',
 ' Equity Shares',
 ' Gold',
 ' Home Loans',
 ' Mutual Funds',
 ' NPS',
 ' National Scholarship Scheme',
 ' PPF',
 ' Personal Loans',
 ' Real Estate',
 ' Real Estate Investment Trusts (REITs)',
 ' Savings Account',
 ' Short-Term Mutual Funds',
 ' Stand-Up India Scheme',
 ' Stocks',
 ' Travel Insurance',
 "Children's Mutual Funds",
 'Mutual Funds',
 'National Scholarship Scheme',
 'Personal Loans',
 'Real Estate Investment Trusts (REITs)',
 'Stand-Up India Scheme',
 'Sukanya Samriddhi Yojana',
 'Travel Insurance'}

In [10]:

for row in df['Recommended Investment Products']:
    print(row.split(","))
    

['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['Real Estate Investment Trusts (REITs)', ' Home Loans', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Sukanya Samriddhi Yojana', " Children's Mutual Funds", ' Real Estate Investment Trusts (REITs)', ' Home Loans', ' Stocks']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['Mutual Funds', ' PPF', ' NPS', ' Stocks', ' Real Estate']
["Children's Mutual Funds", ' Personal Loans', ' Consumer Stocks', ' Gold', ' Mutual Funds']
['National Scholarship Scheme', ' Education Bonds', ' Education Loans', ' Stocks', ' Mutual Funds']
['National Scholarship Scheme', ' Education Bonds', ' Educ

In [11]:
# Split the products and explode → one product per row, keeping all other columns duplicated
df_long = (
    df['Recommended Investment Products']
    .str.split(', ')
    .explode()
    .explode()
    .to_frame('Recommended Investment Product')
    .join(df.drop('Recommended Investment Products', axis=1))
    .reset_index(drop=True)
)

df_long

,Recommended Investment Product,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy
0,Mutual Funds,saving for retirement,44,Male,Medium,2
1,PPF,saving for retirement,44,Male,Medium,2
2,NPS,saving for retirement,44,Male,Medium,2
3,Stocks,saving for retirement,44,Male,Medium,2
4,Real Estate,saving for retirement,44,Male,Medium,2
...,...,...,...,...,...,...
49063,Personal Loans,purchasing for self,71,Male,Medium,1
49064,Consumer Stocks,purchasing for self,71,Male,Medium,1
49065,Gold,purchasing for self,71,Male,Medium,1
49066,Mutual Funds,purchasing for self,71,Male,Medium,1


In [12]:
df_long.shape

(49068, 6)

In [13]:
all_products = df_long['Recommended Investment Product'].unique()
person_cols = ['Individual Goals', 'Age', 'Gender', 'Risk Tolerance', 'Financial Literacy']

In [14]:
df = df_long.copy()

# Create binary target (1 = this product was recommended for this person)
# df['rec'] = 1

In [15]:
# Label-encode ALL categorical columns (strings → integers)
categorical_columns = ['Individual Goals', 'Gender', 'Risk Tolerance', 'Recommended Investment Product']

label_encoders = {}   # Save them so we can reuse during inference!

for col in categorical_columns:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le                     # ← keep for later use
    print(f"{col:30} → {len(le.classes_)} unique values")



Individual Goals               → 6 unique values
Gender                         → 2 unique values
Risk Tolerance                 → 3 unique values
Recommended Investment Product → 22 unique values


In [16]:
df

,Recommended Investment Product,Individual Goals,Age,Gender,Risk Tolerance,Financial Literacy,Individual Goals_encoded,Gender_encoded,Risk Tolerance_encoded,Recommended Investment Product_encoded
0,Mutual Funds,saving for retirement,44,Male,Medium,2,4,1,2,9
1,PPF,saving for retirement,44,Male,Medium,2,4,1,2,12
2,NPS,saving for retirement,44,Male,Medium,2,4,1,2,10
3,Stocks,saving for retirement,44,Male,Medium,2,4,1,2,19
4,Real Estate,saving for retirement,44,Male,Medium,2,4,1,2,14
...,...,...,...,...,...,...,...,...,...,...
49063,Personal Loans,purchasing for self,71,Male,Medium,1,2,1,2,13
49064,Consumer Stocks,purchasing for self,71,Male,Medium,1,2,1,2,2
49065,Gold,purchasing for self,71,Male,Medium,1,2,1,2,7
49066,Mutual Funds,purchasing for self,71,Male,Medium,1,2,1,2,9


In [17]:
feature_cols = [
    'Individual Goals_encoded',
    'Gender_encoded', 
    'Risk Tolerance_encoded',
    'Age',
    'Financial Literacy'
]

X = df[feature_cols]
y = df['Recommended Investment Product_encoded']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [18]:
early_stop = EarlyStopping(rounds=50, metric_name='auc', save_best=True)

model = xgb.XGBClassifier(
    n_estimators=2000,           # we let early stopping cut it short
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    tree_method='hist',
    n_jobs=-1,
    verbosity=1,
    callbacks=[early_stop]       # ← correct place
)

# IMPORTANT: eval_set must be a list of (X, y) tuples — NOT named tuples!
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],   # ← just (X_val, y_val), no string name!
    verbose=50
)

print("Training finished!")
print("Best iteration :", model.get_booster().best_iteration)
print("Best AUC        :", model.get_booster().best_score)

[0]	validation_0-auc:0.78383
[50]	validation_0-auc:0.77902
[52]	validation_0-auc:0.77769
Training finished!
Best iteration : 3
Best AUC        : 0.8290575817999358


In [19]:
# TOP-5 RECOMMENDATION FUNCTION 
def recommend_top5(age, gender, risk_tolerance, financial_literacy, individual_goals):
    
    # Encode the input using the SAME label encoders we saved during training
    goals_enc     = label_encoders['Individual Goals'].transform([individual_goals])[0]
    gender_enc    = label_encoders['Gender'].transform([gender])[0]
    risk_enc      = label_encoders['Risk Tolerance'].transform([risk_tolerance])[0]
    
    # All possible products
    all_products = label_encoders['Recommended Investment Product'].classes_
    
    # Build candidate rows (one per product)
    candidates = []
    for prod in all_products:
        prod_enc = label_encoders['Recommended Investment Product'].transform([prod])[0]
        candidates.append([
            goals_enc,
            gender_enc,
            risk_enc,
            age,
            financial_literacy
        ])
    
    candidates_df = pd.DataFrame(
        candidates,
        columns=feature_cols,
        index=all_products
    )
    
    # Predict probability
    probs = model.predict_proba(candidates_df)[:, 1]
    
    # Rank and return top 5
    
    # result = pd.DataFrame({
    #     'Recommended Investment Product': all_products,
    # }).head(5)
    
    result = pd.DataFrame({
        'Recommended Investment Product': all_products,
        'probability': probs
    }).sort_values('probability', ascending=False).head(5).reset_index(drop=True)
    
    return result



In [ ]:
import pandas as pd
import requests
import json

# Make sure you already have your recommend_top5() function and model trained!
# (from the previous working script)

# Call your model
top5 = recommend_top5(
    age=35,
    gender='Male',
    risk_tolerance='High',
    financial_literacy=4,
    individual_goals='saving for retirement'
)

products = top5['Recommended Investment Product'].tolist()
probs = (top5['probability'] * 100).round(1).astype(str) + "%"

# Build friendly prompt
prompt = f"""
You are a warm, trusted financial advisor speaking to a 35-year-old man who wants to save for retirement.
He is comfortable with high risk and has good financial knowledge.

Here are the top 5 investments I recommend for him:

1. {products[0]} — best match ({probs[0]} confidence)
2. {products[1]} — very strong ({probs[1]})
3. {products[2]} — excellent fit ({probs[2]})
4. {products[3]} — solid choice ({probs[3]})
5. {products[4]} — good addition ({probs[4]})

Please explain each one in 2–3 simple, encouraging sentences using "you".
Start with: "Great news! Based on your profile, here are the 5 best investments for your retirement goal:"
End with: "You're in a fantastic position — let's get started whenever you're ready!"
"""

# Call Ollama (completely free & local)
def ask_ollama(prompt, model="llama3.1:8b"):   # or "phi3:mini"
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        }
    )
    if response.status_code == 200:
        return response.json()['message']['content']
    else:
        return f"Error: {response.status_code} — {response.text}"

print("\n" + "═"*70)
print("YOUR PERSONAL FINANCIAL ADVISOR (FREE & LOCAL)")
print("═"*70)
explanation = ask_ollama(prompt, model="llama3.1:8b")   # or "phi3:mini"
print(explanation)


══════════════════════════════════════════════════════════════════════
YOUR PERSONAL FINANCIAL ADVISOR (FREE & LOCAL)
══════════════════════════════════════════════════════════════════════
Error: 404 — {"error":"model 'llama3.1:8b' not found"}
